# 7: Trajectory Purpose Classification

Assigns each trajectory the purpose with the largest share of its meaningful dwell (Stage 7). POI classes map to purposes through `skos:related`, directly or through a superclass. Start and destination interactions count with the minimum meaningful dwell of their POI type, since their dwell is not observed. A trajectory with no meaningful interaction falls back to TransportationPurpose.


In [ ]:
from helpers.fuseki_config import update_endpoint
from helpers.fuseki_queries import execute_sparql_query, execute_sparql_update, format_results, load_query

In [ ]:
import pandas as pd
import math
from helpers.fuseki_config import query_endpoint

# Everything here runs against the non-inference endpoint. The original
# interaction-purpose-indication.sparql leaned on the OWLMicro reasoner for
# rdfs:subClassOf* closure, which is what made it slow — we do the closure
# in Python over a few hundred ontology triples instead.

# --- 1. Fetch raw interaction facts (no inference) ---------------------------
interactions_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX trajecto: <http://w3id.org/trajecto#>
    PREFIX poi:      <http://w3id.org/trajecto/poi#>
    PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
    SELECT DISTINCT ?interaction ?trajectory ?seconds ?poi WHERE {
        ?interaction a poi:MeaningfulInteraction .
        ?position trajecto:hasInteraction ?interaction ;
                  dul:isPartOf ?trajectory .
        ?interaction poi:hasDwellTimeInSeconds ?seconds ;
                     poi:interactsWith ?poi .
    }
"""))
interactions_df['seconds'] = interactions_df['seconds'].astype(float)
print(f'Meaningful interactions: {len(interactions_df)}')

# --- 2. POI → direct class (no inference) ------------------------------------
poi_class_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX trajecto: <http://w3id.org/trajecto#>
    PREFIX poi:      <http://w3id.org/trajecto/poi#>
    SELECT ?poi ?cls WHERE {
        ?poi a ?cls .
        FILTER(?cls != trajecto:PointOfInterest)
        FILTER(?cls != poi:SignificantPOI)
        FILTER(!isBlank(?cls))
    }
"""))

# --- 3. Class → broader (direct subClassOf, ontology only) -------------------
subclass_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?sub ?sup WHERE { ?sub rdfs:subClassOf ?sup . FILTER(!isBlank(?sup)) }
"""))
parent_of_class = {}
for _, r in subclass_df.iterrows():
    parent_of_class.setdefault(r['sub'], []).append(r['sup'])

# --- 4. Class → purpose (skos:related) ---------------------------------------
related_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?cls ?purpose WHERE { ?cls skos:related ?purpose . }
"""))
related = dict(zip(related_df['cls'], related_df['purpose']))

# --- 5. Resolve each POI class to its skos-related purpose by walking parents
def class_to_purpose(cls, depth=20):
    cur, seen = cls, set()
    while cur and cur not in seen and depth > 0:
        if cur in related:
            return related[cur]
        seen.add(cur)
        parents = parent_of_class.get(cur, [])
        cur = parents[0] if parents else None
        depth -= 1
    return None

class_purpose = {c: class_to_purpose(c) for c in poi_class_df['cls'].unique()}

# --- 6b. Endpoints count with the minimum meaningful dwell of their POI type --
# Start/destination interactions are meaningful by position: the recording
# starts or stops at the POI, so their dwell is unobserved (usually 0 s).
# Weighting them by observed dwell would drop them, so they count with the
# threshold they are assumed to meet, or their observed dwell if higher.
import re

def iso_seconds(d):
    m = re.fullmatch(r'P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:([\d.]+)S)?)?', str(d))
    if not m:
        return None
    dd, h, mi, s = (float(x) if x else 0.0 for x in m.groups())
    return dd * 86400 + h * 3600 + mi * 60 + s

cfg_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX poi: <http://w3id.org/trajecto/poi#>
    SELECT ?cfg ?min WHERE {
        ?cfg a poi:POITypeConfiguration ; poi:hasMinimumMeaningfulDuration ?min .
    }
"""))
# Same config-to-class pairing as the Stage 3 dwell rules: a POI class's
# configuration is named <Class>-configuration or <Class>POI-configuration.
POI_NS = 'http://w3id.org/trajecto/poi#'
min_of_class = {}
for _, r in cfg_df.iterrows():
    base = r['cfg'].split('#')[-1][:-len('-configuration')]
    for name in {base, base[:-3] if base.endswith('POI') else base}:
        min_of_class[POI_NS + name] = iso_seconds(r['min'])

def class_min(cls, depth=20):
    cur, seen = cls, set()
    while cur and cur not in seen and depth > 0:
        if min_of_class.get(cur) is not None:
            return min_of_class[cur]
        seen.add(cur)
        parents = parent_of_class.get(cur, [])
        cur = parents[0] if parents else None
        depth -= 1
    return None

poi_min = (poi_class_df.assign(min_s=poi_class_df['cls'].map(class_min))
           .dropna(subset=['min_s']).groupby('poi')['min_s'].max())
is_endpoint = interactions_df['interaction'].str.endswith(('_origin', '_destination'))
threshold = interactions_df.loc[is_endpoint, 'poi'].map(poi_min)
interactions_df.loc[is_endpoint, 'seconds'] = (
    interactions_df.loc[is_endpoint, 'seconds'].where(
        threshold.isna() | (interactions_df.loc[is_endpoint, 'seconds'] >= threshold), threshold))
print(f'Endpoint interactions: {int(is_endpoint.sum())}, '
      f'with a type threshold: {int(threshold.notna().sum())}')

# --- 6. Attach a purpose to each interaction via its POI's class -------------
poi_class_df['purpose'] = poi_class_df['cls'].map(class_purpose)
poi_purpose = (poi_class_df.dropna(subset=['purpose'])
               .drop_duplicates('poi')[['poi', 'purpose']])
interactions_df = interactions_df.merge(poi_purpose, on='poi', how='inner')
print(f'Interactions with resolved purpose: {len(interactions_df)}')

# --- 7. Roll narrow purposes up to top-level SKOS concepts -------------------
tops_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT DISTINCT ?top WHERE { ?top skos:topConceptOf ?scheme . }
"""))
broader_df = format_results(execute_sparql_query(query_endpoint, """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?narrow ?parent WHERE { ?narrow skos:broader ?parent . }
"""))
top_concepts = set(tops_df['top'].tolist())
parent_of = dict(zip(broader_df['narrow'], broader_df['parent']))

def to_top(uri, depth=20):
    cur, seen = uri, set()
    while cur and cur not in seen and depth > 0:
        if cur in top_concepts:
            return cur
        seen.add(cur)
        cur = parent_of.get(cur)
        depth -= 1
    return uri

rollup = {u: to_top(u) for u in interactions_df['purpose'].unique()}
interactions_df['top_purpose'] = interactions_df['purpose'].map(rollup)
interactions_df = interactions_df.dropna(subset=['top_purpose'])

# --- 8. Dominant purpose per trajectory, transport demoted to fallback -------
TRANSPORT_TOP = 'http://w3id.org/trajecto/purpose/skos#TransportationPurpose'

trajectory_totals = interactions_df.groupby('trajectory')['seconds'].sum()
purpose_totals = (interactions_df
                  .groupby(['trajectory', 'top_purpose'])['seconds']
                  .sum()
                  .reset_index(name='total_seconds')
                  .rename(columns={'top_purpose': 'purpose'}))

non_transport = purpose_totals[purpose_totals['purpose'] != TRANSPORT_TOP]
has_non_transport = set(non_transport['trajectory'].unique())

preferred = non_transport[non_transport['trajectory'].isin(has_non_transport)]
fallback = purpose_totals[~purpose_totals['trajectory'].isin(has_non_transport)]
candidates = pd.concat([preferred, fallback], ignore_index=True)

dominant_purposes = (candidates
                     .loc[candidates.groupby('trajectory')['total_seconds'].idxmax()])

print(f'Candidate assertions: {len(dominant_purposes)}')
print(dominant_purposes['purpose'].map(lambda u: str(u).split('#')[-1].split('/')[-1]).value_counts())

# --- 9. Clear stale + batch insert -------------------------------------------
execute_sparql_update(update_endpoint, """
    PREFIX trajecto: <http://w3id.org/trajecto#>
    PREFIX purpose: <http://w3id.org/trajecto/purpose#>
    DELETE { ?t trajecto:hasPurpose ?p . ?t purpose:hasPurposeConfidenceScore ?c . }
    WHERE  { ?t trajecto:hasPurpose ?p . OPTIONAL { ?t purpose:hasPurposeConfidenceScore ?c . } }
""")

def is_bad(v):
    # Guard against NaN/None URIs — SPARQL parser reads `nan` as a prefix and 400s.
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    return str(v).strip() == '' or str(v).lower() == 'nan'

triples, skipped = [], 0
for _, row in dominant_purposes.iterrows():
    total = trajectory_totals.get(row['trajectory'], 0)
    if is_bad(row['purpose']) or is_bad(row['trajectory']) or not total or is_bad(row['total_seconds']):
        skipped += 1
        continue
    conf = (float(row['total_seconds']) / float(total)) * 100
    if math.isnan(conf):
        skipped += 1
        continue
    triples.append(
        f"<{row['trajectory']}> trajecto:hasPurpose <{row['purpose']}> ; "
        # Canonical decimal form (100.0, not 100.00): TDB2 keeps the lexical
        # form as written, so a non-canonical literal survives the DELETE above.
        f"purpose:hasPurposeConfidenceScore {round(conf, 2)!r} ."
    )
print(f'Built {len(triples)} insert triples (skipped {skipped})')

CHUNK = 500
for i in range(0, len(triples), CHUNK):
    body = '\n'.join(triples[i:i+CHUNK])
    execute_sparql_update(update_endpoint, f"""
        PREFIX trajecto: <http://w3id.org/trajecto#>
        PREFIX purpose: <http://w3id.org/trajecto/purpose#>
        INSERT DATA {{ {body} }}
    """)
print('Done.')

In [ ]:
transportation_query = load_query('transportation-purpose.sparql')
execute_sparql_update(update_endpoint, transportation_query)

In [ ]:
# Validate KG state against stage-07-classification SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-07-classification')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-07-classification', halt_on_violations=True)